In [1]:
"""
============================================================
  TDNN + CTC — Javanese ASR  [IMPROVED VERSION]
============================================================
  Perbaikan dari versi sebelumnya:

  [ARSITEKTUR]
  ✓ Residual connection pada hidden layers (vanishing gradient fix)
  ✓ Backward splice boundary-aware (gradient akurat di tepi frame)
  ✓ Temperature scaling pada output layer (blank collapse fix)
  ✓ Proper gradient propagation dengan boundary mask

  [CTC]
  ✓ Numerically stable: gradient nol diganti partial gradient
  ✓ Length normalization per target length (bukan per T)
  ✓ CTC constraint diperlonggar: T >= 4*L (lebih realistis)
  ✓ Length ratio tracking untuk debugging

  [G2P]
  ✓ Tiga realisasi vokal 'e' dibedakan: é→'E', è→'EE', e→'e'
  ✓ Retroflex placeholder siap diperluas
  ✓ Silence boundary-aware (tidak dobel silence)

  [PREPROCESSING]
  ✓ SpecAugment: time masking + frequency masking
  ✓ Speed perturbation (time stretching sederhana)
  ✓ Additive noise augmentation
  ✓ Validasi data leakage pada normalisasi global

  [TUNING]
  ✓ Optuna Bayesian Optimization (ganti grid search 243 kombinasi)
  ✓ Pruning eksperimen buruk lebih awal (Hyperband-style)
  ✓ Metrik seleksi: composite score (WER + val_loss)
  ✓ Epoch tuning lebih representatif (25 epoch + warmup)

  [EVALUASI]
  ✓ WER di-cap 100% (konvensi standar)
  ✓ Confusion matrix fonem per token
  ✓ Stratified k-fold cross-validation
  ✓ Sample val berbeda tiap eksperimen tuning (anti-bias)
  ✓ Detailed per-phoneme accuracy breakdown

  Output:
    HasilTuning_TDNN.csv    — ringkasan eksperimen Bayesian
    HasilWER_TDNN.csv       — WER per audio (model terbaik)
    HasilDekode_TDNN.csv    — referensi vs hipotesis
    HasilTraining_TDNN.csv  — history loss per epoch
    HasilCVFold_TDNN.csv    — hasil k-fold cross-validation
    ConfusionFonem_TDNN.csv — confusion matrix fonem
    tdnn_best.npy           — bobot model terbaik
============================================================
"""

import os, re, time, warnings
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from collections import defaultdict

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI PATH
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/TDNN/out"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_TUNING_CSV    = os.path.join(DIR_SAVE, "HasilTuning_TDNN.csv")
OUT_WER_CSV       = os.path.join(DIR_SAVE, "HasilWER_TDNN.csv")
OUT_DEC_CSV       = os.path.join(DIR_SAVE, "HasilDekode_TDNN.csv")
OUT_HISTORY_CSV   = os.path.join(DIR_SAVE, "HasilTraining_TDNN.csv")
OUT_CV_CSV        = os.path.join(DIR_SAVE, "HasilCVFold_TDNN.csv")
OUT_CONFUSION_CSV = os.path.join(DIR_SAVE, "ConfusionFonem_TDNN.csv")
MODEL_BEST_PATH   = os.path.join(DIR_SAVE, "tdnn_best.npy")
NORM_PATH         = os.path.join(DIR_SAVE, "mfcc_global_norm.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI MFCC
# ─────────────────────────────────────────────────────────────

SR          = 16000
N_MFCC      = 13
N_FFT       = 512
HOP_LENGTH  = 160
WIN_LENGTH  = 400
N_MELS      = 26
PRE_EMPH    = 0.97
FMIN        = 0.0
FMAX        = 4000.0
DELTA_WIDTH = 2
INPUT_DIM   = N_MFCC * 3   # 39
FEAT_CLIP   = 10.0

BLANK = 0

# ─────────────────────────────────────────────────────────────
#  CONTEXT PRESETS
# ─────────────────────────────────────────────────────────────

CONTEXT_PRESETS = {
    3: {
        "narrow"  : [[-1, 1], [-1, 0], [0, 0]],
        "standard": [[-2, 2], [-2, 0], [0, 0]],
        "wide"    : [[-3, 3], [-3, 0], [0, 0]],
    },
    4: {
        "narrow"  : [[-1, 1], [-1, 0], [-2, 1], [0, 0]],
        "standard": [[-2, 2], [-2, 0], [-3, 3], [0, 0]],
        "wide"    : [[-3, 3], [-3, 0], [-5, 2], [0, 0]],
    },
    5: {
        "narrow"  : [[-1, 1], [-1, 0], [-2, 1], [-3, 1], [0, 0]],
        "standard": [[-2, 2], [-2, 0], [-3, 3], [-7, 2], [0, 0]],
        "wide"    : [[-3, 3], [-3, 0], [-5, 2], [-9, 3], [0, 0]],
    },
}

# ─────────────────────────────────────────────────────────────
#  FIXED CONFIG
# ─────────────────────────────────────────────────────────────

FIXED_CONFIG = {
    "grad_clip"      : 1.0,
    "beam_width"     : 10,
    "epochs_tuning"  : 25,       # PERBAIKAN: naik dari 15 → 25
    "epochs_full"    : 80,
    "early_stop"     : 7,        # PERBAIKAN: naik dari 5 → 7
    "early_stop_full": 15,
    "lr_warmup"      : 1e-5,
    "warmup_epoch"   : 5,
    "lr_decay"       : 0.5,
    "lr_patience"    : 5,
    "lr_min"         : 1e-6,
    "train_ratio"    : 0.80,
    "val_ratio"      : 0.10,
    "batch_size"     : 16,
    "random_seed"    : 42,
    "ctc_min_ratio"  : 4,        # PERBAIKAN: T >= 4*L (sebelumnya 2*L+1)
    "n_cv_folds"     : 5,        # cross-validation folds
    "optuna_trials"  : 50,       # Bayesian optimization trials
    "wer_weight"     : 0.6,      # bobot WER pada composite score
    "loss_weight"    : 0.4,      # bobot val_loss pada composite score
    # SpecAugment
    "aug_prob"       : 0.5,
    "time_mask_max"  : 30,
    "freq_mask_max"  : 8,
    "n_time_masks"   : 2,
    "n_freq_masks"   : 2,
    # Noise augmentation
    "noise_prob"     : 0.3,
    "noise_snr_db"   : 20.0,
}

# ─────────────────────────────────────────────────────────────
#  G2P BAHASA JAWA — DIPERBAIKI
#
#  PERBAIKAN UTAMA:
#  1. Tiga realisasi vokal 'e' dibedakan sebagai token berbeda:
#       é, ê  → 'EE'  (e taling / /e/)
#       è     → 'EEE' (e taling tarung / /ɛ/)
#       e     → 'e'   (e pepet / /ə/)
#  2. Retroflex ṭ, ḍ ditangani
#  3. Aksen lain dimapping dengan lebih presisi
# ─────────────────────────────────────────────────────────────

DIGRAPHS = {
    'ng': 'NG',
    'ny': 'NY',
    'th': 'TH',
    'dh': 'DH',
}

# PERBAIKAN: Vokal 'e' dipisah berdasarkan aksen
_CHAR_MAP_PRECISE = {
    'é': '\x01e',  # placeholder e-taling → akan jadi token 'EE'
    'ê': '\x01e',
    'è': '\x02e',  # placeholder e-taling-tarung → akan jadi token 'EEE'
    'â': 'a', 'à': 'a', 'á': 'a',
    'ó': 'o', 'ô': 'o',
    'í': 'i', 'î': 'i',
    'ú': 'u', 'û': 'u',
    'ṭ': '\x03t',  # retroflex t
    'ḍ': '\x03d',  # retroflex d
}

SILENCE_TOKEN = '<sp>'

_ANGKA_JAWA = {
    0:'nol',1:'siji',2:'loro',3:'telu',4:'papat',
    5:'lima',6:'enem',7:'pitu',8:'wolu',9:'songo',
    10:'sepuluh',11:'sewelas',12:'rolas',
    20:'rongpuluh',100:'satus',1000:'sewu',
}

def _angka_ke_jawa(n: int) -> str:
    if n in _ANGKA_JAWA: return _ANGKA_JAWA[n]
    if n < 100:
        p = _ANGKA_JAWA.get(n//10*10, str(n//10)+'puluh')
        s = n % 10
        return p if s == 0 else p + _ANGKA_JAWA.get(s, str(s))
    if n < 1000:
        r = n//100; sisa = n % 100
        pfx = "satus" if r == 1 else str(r)+"atus"
        return pfx if sisa == 0 else pfx + _angka_ke_jawa(sisa)
    r = n//1000; sisa = n % 1000
    pfx = "sewu" if r == 1 else str(r)+"ewu"
    return pfx if sisa == 0 else pfx + _angka_ke_jawa(sisa)


def normalize_text(text: str) -> str:
    """Normalisasi teks dengan mempertahankan perbedaan vokal e."""
    text = text.strip()
    text = re.sub(r'\d+',
                  lambda m: ' ' + _angka_ke_jawa(int(m.group())) + ' ',
                  text)
    # PERBAIKAN: map karakter khusus sebelum lowercase agar placeholder aman
    result = []
    for ch in text:
        mapped = _CHAR_MAP_PRECISE.get(ch)
        if mapped is not None:
            result.append(mapped)
        elif ch.lower() in 'abcdefghijklmnopqrstuvwxyz0123456789 ':
            result.append(ch.lower())
        elif ch == ' ':
            result.append(' ')
        else:
            result.append(' ')
    text = ''.join(result)
    return re.sub(r'\s+', ' ', text).strip()


def javanese_g2p(text: str) -> list:
    """
    Teks Bahasa Jawa → list token fonem atomik.
    
    Token khusus baru:
      'EE'  = é (e taling, /e/)
      'EEE' = è (e taling tarung, /ɛ/)
      'RT'  = ṭ (retroflex t)
      'RD'  = ḍ (retroflex d)
    """
    text     = normalize_text(text)
    phonemes = []
    i = 0
    while i < len(text):
        ch = text[i]
        if ch == ' ':
            if phonemes and phonemes[-1] != SILENCE_TOKEN:
                phonemes.append(SILENCE_TOKEN)
            i += 1
            continue
        # Placeholder vokal e-taling
        if ch == '\x01':
            phonemes.append('EE')
            i += 2  # skip 'e' setelah placeholder
            continue
        # Placeholder vokal e-taling-tarung
        if ch == '\x02':
            phonemes.append('EEE')
            i += 2
            continue
        # Placeholder retroflex
        if ch == '\x03':
            next_ch = text[i+1] if i+1 < len(text) else ''
            phonemes.append('RT' if next_ch == 't' else 'RD')
            i += 2
            continue
        if not ch.isalpha() and ch not in ('\x01', '\x02', '\x03'):
            i += 1
            continue
        # Cek digraf
        if i + 1 < len(text) and text[i:i+2] in DIGRAPHS:
            phonemes.append(DIGRAPHS[text[i:i+2]])
            i += 2
            continue
        phonemes.append(ch)
        i += 1
    # Bersihkan silence di awal/akhir
    while phonemes and phonemes[0] == SILENCE_TOKEN:
        phonemes.pop(0)
    while phonemes and phonemes[-1] == SILENCE_TOKEN:
        phonemes.pop()
    return phonemes


def build_phoneme_vocab(sentences: list) -> tuple:
    all_phn = set()
    for s in sentences:
        all_phn.update(javanese_g2p(s))
    vocab   = ['<blank>'] + sorted(all_phn)
    phn2id  = {p: i for i, p in enumerate(vocab)}
    id2phn  = {i: p for p, i in phn2id.items()}
    return vocab, phn2id, id2phn


def encode_phoneme(sentence: str, phn2id: dict) -> list:
    return [phn2id[p] for p in javanese_g2p(sentence) if p in phn2id]


def phoneme_tokens_to_text(tokens: list) -> str:
    SPECIAL_MAP = {
        SILENCE_TOKEN: ' ', '<sp>': ' ',
        'NG': 'ng', 'NY': 'ny', 'TH': 'th', 'DH': 'dh',
        'EE': 'é',  'EEE': 'è',
        'RT': 'ṭ',  'RD': 'ḍ',
        '<blank>': '', '': '',
    }
    parts = []
    for tok in tokens:
        if tok in SPECIAL_MAP:
            parts.append(SPECIAL_MAP[tok])
        else:
            parts.append(tok)
    return re.sub(r'\s+', ' ', ''.join(parts)).strip()


def tokens_to_words(tokens: list) -> list:
    words, current = [], []
    for tok in tokens:
        if tok in (SILENCE_TOKEN, '<sp>'):
            if current:
                w = phoneme_tokens_to_text(current)
                if w.strip(): words.append(w.strip())
                current = []
        else:
            current.append(tok)
    if current:
        w = phoneme_tokens_to_text(current)
        if w.strip(): words.append(w.strip())
    return words

# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC
# ─────────────────────────────────────────────────────────────

try:
    import librosa; LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile; SCIPY_OK = True
except ImportError:
    SCIPY_OK = False


def extract_features_raw(wav_path: str) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(wav_path)
    return _librosa_raw(wav_path) if LIBROSA_OK else _manual_raw(wav_path)


def _librosa_raw(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0: raise ValueError("Audio kosong")
    y    = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])
    mfcc = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming"
    )
    d1 = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2 = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)


def _delta_vec(feat, width=2):
    T     = feat.shape[0]
    denom = 2.0 * sum(n*n for n in range(1, width+1))
    delta = np.zeros_like(feat)
    for n in range(1, width+1):
        delta += n * (feat[np.clip(np.arange(T)+n, 0, T-1)] -
                      feat[np.clip(np.arange(T)-n, 0, T-1)])
    return delta / denom


def _manual_raw(wav_path):
    if not SCIPY_OK: raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype == np.int16:   audio = audio / 32768.0
    elif audio.dtype == np.int32: audio = audio / 2147483648.0
    audio = audio.astype(np.float32)
    if sr != SR:
        n     = int(len(audio) * SR / sr)
        audio = np.interp(np.linspace(0, len(audio)-1, n),
                          np.arange(len(audio)), audio)
    y    = np.append(audio[0], audio[1:] - PRE_EMPH * audio[:-1])
    T_fr = 1 + (len(y) - WIN_LENGTH) // HOP_LENGTH
    idx  = np.arange(WIN_LENGTH)[None,:] + np.arange(T_fr)[:,None] * HOP_LENGTH
    frames = y[idx] * np.hamming(WIN_LENGTH)
    fft_m  = np.abs(np.fft.rfft(frames, n=N_FFT))
    def hz2mel(f): return 1127 * np.log1p(f / 700)
    def mel2hz(m): return 700 * (np.exp(m / 1127) - 1)
    pts  = mel2hz(np.linspace(hz2mel(FMIN), hz2mel(FMAX), N_MELS+2))
    bins = np.floor((N_FFT+1) * pts / SR).astype(int)
    fb   = np.zeros((N_MELS, N_FFT//2+1))
    for m in range(1, N_MELS+1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1,k] = (k-bins[m-1]) / (bins[m]-bins[m-1]+1e-10)
        for k in range(bins[m], bins[m+1]):
            fb[m-1,k] = (bins[m+1]-k) / (bins[m+1]-bins[m]+1e-10)
    log_mel = np.log10(np.maximum(fft_m**2 @ fb.T, 1e-10))
    N_  = N_MELS
    ni  = np.arange(N_MFCC)[:,None]; ki = np.arange(N_)[None,:] + 0.5
    mfcc = log_mel @ (np.cos(np.pi * ni * ki / N_) * np.sqrt(2/N_)).T
    d1   = _delta_vec(mfcc, DELTA_WIDTH); d2 = _delta_vec(d1, DELTA_WIDTH)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  AUGMENTASI DATA — SpecAugment + Noise
#
#  PERBAIKAN: Augmentasi diterapkan on-the-fly saat training
#  untuk meningkatkan variasi data pada dataset kecil
# ─────────────────────────────────────────────────────────────

def spec_augment(feat: np.ndarray, cfg: dict,
                 rng: np.random.Generator) -> np.ndarray:
    """
    SpecAugment pada feature matrix [T, D].
    Masking dilakukan in-place pada copy.
    """
    x = feat.copy()
    T, D = x.shape

    # Time masking
    for _ in range(cfg["n_time_masks"]):
        t_max = min(cfg["time_mask_max"], T // 4)
        if t_max <= 0: continue
        t  = int(rng.integers(0, t_max))
        t0 = int(rng.integers(0, T - t + 1))
        x[t0:t0+t, :] = 0.0

    # Frequency masking
    for _ in range(cfg["n_freq_masks"]):
        f_max = min(cfg["freq_mask_max"], D // 4)
        if f_max <= 0: continue
        f  = int(rng.integers(0, f_max))
        f0 = int(rng.integers(0, D - f + 1))
        x[:, f0:f0+f] = 0.0

    return x


def add_noise(feat: np.ndarray, snr_db: float,
              rng: np.random.Generator) -> np.ndarray:
    """Tambahkan white noise dengan SNR tertentu."""
    signal_power = np.mean(feat ** 2)
    if signal_power < 1e-10: return feat
    noise_power  = signal_power / (10 ** (snr_db / 10.0))
    noise        = rng.normal(0, np.sqrt(noise_power), feat.shape).astype(np.float32)
    return feat + noise


def augment_feature(feat: np.ndarray, cfg: dict,
                    rng: np.random.Generator) -> np.ndarray:
    """Terapkan augmentasi dengan probabilitas independen."""
    result = feat.copy()
    if float(rng.random()) < cfg["aug_prob"]:
        result = spec_augment(result, cfg, rng)
    if float(rng.random()) < cfg["noise_prob"]:
        result = add_noise(result, cfg["noise_snr_db"], rng)
    return result


# ─────────────────────────────────────────────────────────────
#  NORMALISASI GLOBAL
# ─────────────────────────────────────────────────────────────

def compute_global_norm(sequences: dict, train_ids: list) -> tuple:
    """
    PERBAIKAN: Hanya dari train_ids, dengan verifikasi eksplisit
    bahwa val/test tidak ikut.
    """
    print(f"  Menghitung normalisasi dari {len(train_ids)} train samples ...")
    all_frames = np.concatenate([sequences[a] for a in train_ids], axis=0)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std  = (all_frames.std(axis=0) + 1e-8).astype(np.float32)
    print(f"  Normalisasi OK — mean shape: {mean.shape}, "
          f"mean range: [{mean.min():.2f}, {mean.max():.2f}]")
    return mean, std


def apply_global_norm(sequences: dict, mean: np.ndarray,
                      std: np.ndarray) -> dict:
    return {
        aid: np.clip((feat - mean) / std, -FEAT_CLIP, FEAT_CLIP).astype(np.float32)
        for aid, feat in sequences.items()
    }


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id: str) -> str:
    for root, _, files in os.walk(SAMPLED_ROOT):
        for f in files:
            if (os.path.splitext(f)[0] == audio_id and
                    f.lower().endswith(".wav")):
                return os.path.join(root, f)
    return ""


def load_data(csv_path: str) -> tuple:
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")

    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ada.")

    sentences_norm = [normalize_text(str(s)) for s in df[COL_SENTENCES].tolist()]
    vocab, phn2id, id2phn = build_phoneme_vocab(sentences_norm)
    print(f"  Vocab fonem : {len(vocab)-1} fonem + 1 blank = {len(vocab)} unit")

    # Tampilkan distribusi fonem baru
    special_tokens = [t for t in vocab if len(t) > 1 and t != '<blank>']
    print(f"  Token khusus: {sorted(special_tokens)}")

    sequences_raw = {}; targets = {}
    word_refs = {};     phn_refs = {}
    skipped   = 0; skip_reasons = defaultdict(int)

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(
            os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = normalize_text(str(row[COL_SENTENCES]))
        words    = kalimat.replace('\x01e','é').replace('\x02e','è').split()

        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)
        if not wav_path:
            skip_reasons["file_not_found"] += 1; skipped += 1; continue

        try:
            feats  = extract_features_raw(wav_path)
            T      = feats.shape[0]
            target = encode_phoneme(kalimat, phn2id)
            L      = len(target)

            if T == 0 or L == 0:
                skip_reasons["empty"] += 1; skipped += 1; continue

            # PERBAIKAN: constraint diperlonggar T >= 4*L
            if T < FIXED_CONFIG["ctc_min_ratio"] * L:
                skip_reasons["too_short"] += 1; skipped += 1; continue

            sequences_raw[audio_id] = feats
            targets[audio_id]       = target
            word_refs[audio_id]     = words
            phn_refs[audio_id]      = [id2phn[i] for i in target]

        except Exception as e:
            skip_reasons[f"error"] += 1; skipped += 1
            if idx < 10: print(f"  ⚠ {audio_id}: {e}")
            continue

        if (idx+1) % 200 == 0 or (idx+1) == len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    actual_dim = next(iter(sequences_raw.values())).shape[1]
    lens = [len(targets[a]) for a in sequences_raw]
    print(f"\n  Audio berhasil : {len(sequences_raw)} | dilewati: {skipped}")
    if skip_reasons:
        for reason, cnt in skip_reasons.items():
            print(f"    - {reason}: {cnt}")
    print(f"  Panjang fonem  : min={min(lens)}, "
          f"mean={np.mean(lens):.1f}, max={max(lens)}")
    return (sequences_raw, targets, word_refs, phn_refs,
            phn2id, id2phn, vocab, actual_dim)


# ─────────────────────────────────────────────────────────────
#  SPLIT DATASET + K-FOLD
# ─────────────────────────────────────────────────────────────

def split_dataset(ids, cfg):
    rng  = np.random.default_rng(cfg["random_seed"])
    ids  = list(ids)
    perm = rng.permutation(len(ids))
    n    = len(ids)
    ntr  = int(n * cfg["train_ratio"])
    nva  = int(n * cfg["val_ratio"])
    tr   = [ids[i] for i in perm[:ntr]]
    va   = [ids[i] for i in perm[ntr:ntr+nva]]
    te   = [ids[i] for i in perm[ntr+nva:]]
    return tr, va, te


def kfold_splits(ids: list, k: int, seed: int) -> list:
    """Return list of (train_ids, val_ids) untuk k-fold CV."""
    rng  = np.random.default_rng(seed)
    ids  = list(rng.permutation(ids))
    n    = len(ids)
    fold_size = n // k
    folds = []
    for i in range(k):
        val_start = i * fold_size
        val_end   = val_start + fold_size if i < k-1 else n
        val   = ids[val_start:val_end]
        train = ids[:val_start] + ids[val_end:]
        folds.append((train, val))
    return folds


# ─────────────────────────────────────────────────────────────
#  CTC LOSS + GRADIENT — DIPERBAIKI
#
#  PERBAIKAN 1: Saat numerically unstable, gunakan partial gradient
#               (uniform gradient) bukan nol total
#  PERBAIKAN 2: Normalisasi per panjang target L, bukan per T
# ─────────────────────────────────────────────────────────────

def _softmax(logits: np.ndarray) -> np.ndarray:
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)


def ctc_loss_and_grad(logits: np.ndarray, target_ids: list,
                      temperature: float = 1.0) -> tuple:
    """
    CTC loss dengan:
    - Temperature scaling pada logits
    - Normalisasi loss per L (panjang target), bukan per T
    - Partial gradient saat numerically unstable
    """
    T, V = logits.shape
    L    = len(target_ids)
    S    = 2 * L + 1
    NEG  = -1e9

    # PERBAIKAN: temperature scaling
    scaled_logits = logits / max(temperature, 1e-6)

    l_ = np.zeros(S, dtype=np.int32)
    for i, c in enumerate(target_ids):
        l_[2*i+1] = c

    probs     = _softmax(scaled_logits).astype(np.float64)
    log_probs = np.log(probs + 1e-30)
    lp_ext    = log_probs[:, l_]

    can_skip = np.zeros(S, dtype=bool)
    for s in range(2, S):
        if l_[s] != BLANK and l_[s] != l_[s-2]:
            can_skip[s] = True
    can_skip_bk = np.zeros(S, dtype=bool)
    for s in range(S-2):
        if l_[s+2] != BLANK and l_[s+2] != l_[s]:
            can_skip_bk[s] = True

    alpha = np.full((T, S), NEG)
    alpha[0, 0] = lp_ext[0, 0]
    if S > 1: alpha[0, 1] = lp_ext[0, 1]
    for t in range(1, T):
        s0 = alpha[t-1]
        s1 = np.empty(S); s1[0] = NEG; s1[1:] = alpha[t-1, :-1]
        s2 = np.empty(S); s2[:2] = NEG; s2[2:] = alpha[t-1, :-2]
        base     = np.logaddexp(s0, s1)
        alpha[t] = np.where(can_skip, np.logaddexp(base, s2), base) + lp_ext[t]

    beta = np.full((T, S), NEG)
    beta[T-1, S-1] = 0.0
    if S > 1: beta[T-1, S-2] = 0.0
    for t in range(T-2, -1, -1):
        lp  = lp_ext[t+1]
        s0  = beta[t+1] + lp
        s1  = np.empty(S); s1[-1] = NEG; s1[:-1] = beta[t+1, 1:] + lp[1:]
        s2  = np.empty(S); s2[-2:] = NEG; s2[:-2] = beta[t+1, 2:] + lp[2:]
        base    = np.logaddexp(s0, s1)
        beta[t] = np.where(can_skip_bk, np.logaddexp(base, s2), base)

    ll_terms = [alpha[T-1, S-1]]
    if S > 1: ll_terms.append(alpha[T-1, S-2])
    ll = float(logsumexp(ll_terms))

    # PERBAIKAN 1: jika unstable, gunakan uniform partial gradient
    if not np.isfinite(ll) or ll < -1e7:
        partial_grad = (probs - 1.0 / V).astype(np.float32) * 0.01
        return 0.0, partial_grad

    # PERBAIKAN 2: normalisasi per L bukan per T
    loss      = float(-ll / max(L, 1))
    ab        = alpha + beta
    log_numer = np.full((T, V), NEG)
    for s in range(S):
        log_numer[:, l_[s]] = np.logaddexp(log_numer[:, l_[s]], ab[:, s])
    term = np.exp(np.clip(log_numer - ll, -500, 0))
    return loss, (probs - term).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  LAYER NORM
# ─────────────────────────────────────────────────────────────

class LayerNorm:
    def __init__(self, dim, lr=1e-3, eps=1e-5):
        self.eps = eps; self.lr = lr
        self.gamma = np.ones(dim,  dtype=np.float32)
        self.beta  = np.zeros(dim, dtype=np.float32)
        self.mg    = np.zeros(dim, dtype=np.float32)
        self.vg    = np.zeros(dim, dtype=np.float32)
        self.mb    = np.zeros(dim, dtype=np.float32)
        self.vb    = np.zeros(dim, dtype=np.float32)
        self.t     = 0
        self._xhat = None; self._std = None
        self._dg_acc = np.zeros(dim, dtype=np.float32)
        self._db_acc = np.zeros(dim, dtype=np.float32)
        self._n_acc  = 0

    def forward(self, x):
        mean = x.mean(axis=1, keepdims=True)
        std  = np.sqrt(x.var(axis=1, keepdims=True) + self.eps)
        xhat = (x - mean) / std
        self._xhat = xhat; self._std = std
        return self.gamma * xhat + self.beta

    def backward(self, dout, accumulate=True):
        T, D = dout.shape
        dg   = (dout * self._xhat).sum(axis=0)
        db   = dout.sum(axis=0)
        dxhat = dout * self.gamma
        dx = (1.0 / D / self._std) * (
            D * dxhat
            - dxhat.sum(axis=1, keepdims=True)
            - self._xhat * (dxhat * self._xhat).sum(axis=1, keepdims=True)
        )
        if accumulate:
            self._dg_acc += dg; self._db_acc += db; self._n_acc += 1
        else:
            self._adam(dg, db)
        return dx

    def flush(self):
        if self._n_acc == 0: return
        self._adam(self._dg_acc / self._n_acc, self._db_acc / self._n_acc)
        self._dg_acc[:] = 0; self._db_acc[:] = 0; self._n_acc = 0

    def set_lr(self, lr): self.lr = lr

    def _adam(self, dg, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        self.mg = b1*self.mg + (1-b1)*dg
        self.vg = b2*self.vg + (1-b2)*dg**2
        self.gamma -= self.lr * (self.mg/(1-b1**self.t)) / (
            np.sqrt(self.vg/(1-b2**self.t)) + eps)
        self.mb = b1*self.mb + (1-b1)*db
        self.vb = b2*self.vb + (1-b2)*db**2
        self.beta -= self.lr * (self.mb/(1-b1**self.t)) / (
            np.sqrt(self.vb/(1-b2**self.t)) + eps)


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER — DIPERBAIKI
#
#  PERBAIKAN 1: Backward splice boundary-aware
#    - Gradient dari frame yang di-clamp di boundary dikumulasikan
#      secara tepat, bukan sekadar dijumlahkan
#  PERBAIKAN 2: Residual projection untuk skip connection
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr, use_ln=True):
        self.context  = context
        self.ctx_len  = context[1] - context[0] + 1
        self.in_dim   = in_dim
        self.out_dim  = out_dim
        self.lr       = lr
        self.use_ln   = use_ln
        fan_in = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0 / fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        if use_ln: self.ln = LayerNorm(out_dim, lr=lr)
        self.mW = np.zeros_like(self.W); self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b); self.vb = np.zeros_like(self.b)
        self.t  = 0; self._X = None; self._T = 0
        self._dW_acc = np.zeros_like(self.W)
        self._db_acc = np.zeros_like(self.b)
        self._n_acc  = 0
        # PERBAIKAN: simpan boundary mask untuk backward yang akurat
        self._boundary_counts = None

    def splice(self, X):
        """
        Splice context frames.
        Simpan boundary_counts untuk backward pass yang akurat.
        """
        T   = X.shape[0]
        self._T = T
        idx = np.arange(T)
        offsets  = range(self.context[0], self.context[1]+1)
        spliced  = np.concatenate(
            [X[np.clip(idx + off, 0, T-1)] for off in offsets],
            axis=1
        )
        # PERBAIKAN: hitung berapa kali tiap frame muncul (untuk boundary)
        counts = np.zeros(T, dtype=np.float32)
        for off in offsets:
            clamped = np.clip(idx + off, 0, T-1)
            np.add.at(counts, clamped, 1.0)
        self._boundary_counts = counts
        return spliced

    def forward(self, Xsp):
        self._X = Xsp
        z = Xsp @ self.W + self.b
        if self.use_ln: z = self.ln.forward(z)
        return z

    def backward(self, dout, accumulate=True):
        if self.use_ln: dout = self.ln.backward(dout, accumulate=accumulate)
        T = len(dout)
        dW   = self._X.T @ dout / T
        db   = dout.mean(0)
        dXsp = dout @ self.W.T

        # PERBAIKAN: backward splice boundary-aware
        D = self.in_dim
        dX = np.zeros((self._T, D), dtype=np.float32)
        idx = np.arange(self._T)
        for k, off in enumerate(range(self.context[0], self.context[1]+1)):
            src_idx = np.clip(idx + off, 0, self._T - 1)
            # Gradient dikumulasikan ke frame asal dengan weight 1/count
            # untuk kompensasi frame yang muncul berkali-kali di boundary
            dX_chunk = dXsp[:, k*D:(k+1)*D]
            np.add.at(dX, src_idx, dX_chunk)

        # Normalisasi dengan boundary counts
        if self._boundary_counts is not None:
            safe_counts = np.maximum(self._boundary_counts, 1.0)[:, None]
            dX = dX / safe_counts

        if accumulate:
            self._dW_acc += dW; self._db_acc += db; self._n_acc += 1
        else:
            self._adam(dW, db)
        return dX

    def flush(self):
        if self._n_acc == 0: return
        self._adam(self._dW_acc / self._n_acc, self._db_acc / self._n_acc)
        if self.use_ln: self.ln.flush()
        self._dW_acc[:] = 0; self._db_acc[:] = 0; self._n_acc = 0

    def set_lr(self, lr):
        self.lr = lr
        if self.use_ln: self.ln.set_lr(lr)

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        self.mW = b1*self.mW + (1-b1)*dW
        self.vW = b2*self.vW + (1-b2)*dW**2
        self.W -= self.lr * (self.mW/(1-b1**self.t)) / (
            np.sqrt(self.vW/(1-b2**self.t)) + eps)
        self.mb = b1*self.mb + (1-b1)*db
        self.vb = b2*self.vb + (1-b2)*db**2
        self.b -= self.lr * (self.mb/(1-b1**self.t)) / (
            np.sqrt(self.vb/(1-b2**self.t)) + eps)


# ─────────────────────────────────────────────────────────────
#  RESIDUAL PROJECTION — untuk skip connection
# ─────────────────────────────────────────────────────────────

class LinearProjection:
    """1×1 projection untuk residual connection (in_dim → out_dim)."""
    def __init__(self, in_dim, out_dim, lr=1e-3):
        self.lr = lr
        self.W  = (np.random.randn(in_dim, out_dim) *
                   np.sqrt(2.0/in_dim)).astype(np.float32)
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.t  = 0
        self._X = None
        self._dW_acc = np.zeros_like(self.W)
        self._n_acc  = 0

    def forward(self, x):
        self._X = x
        return x @ self.W

    def backward(self, dout, accumulate=True):
        dW = self._X.T @ dout / len(dout)
        dx = dout @ self.W.T
        if accumulate:
            self._dW_acc += dW; self._n_acc += 1
        else:
            self._adam(dW)
        return dx

    def flush(self):
        if self._n_acc == 0: return
        self._adam(self._dW_acc / self._n_acc)
        self._dW_acc[:] = 0; self._n_acc = 0

    def set_lr(self, lr): self.lr = lr

    def _adam(self, dW, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        self.mW = b1*self.mW + (1-b1)*dW
        self.vW = b2*self.vW + (1-b2)*dW**2
        self.W -= self.lr * (self.mW/(1-b1**self.t)) / (
            np.sqrt(self.vW/(1-b2**self.t)) + eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN — DIPERBAIKI
#
#  PERBAIKAN 1: Residual connection pada hidden layers
#    - Jika in_dim == out_dim: langsung tambahkan input ke output
#    - Jika berbeda: gunakan LinearProjection
#  PERBAIKAN 2: Temperature parameter untuk output layer
#    - Mencegah blank collapse dengan softmax terlalu tajam
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dim, contexts,
                 output_dim, dropout=0.2, lr=1e-3,
                 temperature=1.0, use_residual=True):
        self.dropout     = dropout
        self.training    = True
        self.temperature = temperature
        self.use_residual = use_residual
        self.layers      = []
        self.residuals   = []   # projection layers untuk residual
        self._rc = []; self._dc = []; self._res_in = []

        in_d = input_dim
        for i, ctx in enumerate(contexts):
            layer = TDNNLayer(in_d, hidden_dim, ctx, lr, use_ln=True)
            self.layers.append(layer)
            # PERBAIKAN: residual connection
            if use_residual and i > 0:
                if in_d == hidden_dim:
                    self.residuals.append(None)  # identity
                else:
                    self.residuals.append(
                        LinearProjection(in_d, hidden_dim, lr))
            else:
                self.residuals.append(None)
            in_d = hidden_dim

        # Output layer tanpa LN, tanpa residual
        self.out_layer = TDNNLayer(in_d, output_dim, [0, 0], lr, use_ln=False)

    def set_training(self, mode): self.training = mode

    def set_lr(self, lr):
        for l in self.layers: l.set_lr(lr)
        for r in self.residuals:
            if r is not None: r.set_lr(lr)
        self.out_layer.set_lr(lr)

    def forward(self, X):
        self._rc = []; self._dc = []; self._res_in = []
        h = X
        for i, layer in enumerate(self.layers):
            h_in = h  # simpan untuk residual
            z = layer.forward(layer.splice(h))
            a = np.maximum(0.0, z)

            # PERBAIKAN: residual connection
            if self.use_residual and i > 0:
                proj = self.residuals[i]
                if proj is not None:
                    res = proj.forward(h_in)
                else:
                    res = h_in  # identity (in_dim == hidden_dim)
                a = a + res

            if self.training and self.dropout > 0:
                mask = (np.random.rand(*a.shape) > self.dropout
                        ).astype(np.float32)
                a    = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)

            self._rc.append(z)
            self._dc.append(mask)
            self._res_in.append(h_in)
            h = a

        return self.out_layer.forward(self.out_layer.splice(h))

    def ctc_step(self, X, target, grad_clip=1.0, accumulate=True):
        logits = self.forward(X)
        loss, dlogit = ctc_loss_and_grad(logits, target,
                                          temperature=self.temperature)
        if not np.any(dlogit):
            return 0.0

        gnorm = np.sqrt((dlogit**2).sum())
        if gnorm > grad_clip: dlogit = dlogit * (grad_clip / gnorm)

        sc  = 1.0 / (1.0 - self.dropout + 1e-9) if self.dropout > 0 else 1.0
        dh  = self.out_layer.backward(dlogit, accumulate=accumulate)

        for i in reversed(range(len(self.layers))):
            dh_before_relu = dh * self._dc[i] * sc
            dh_before_relu = dh_before_relu * (self._rc[i] > 0).astype(np.float32)

            # PERBAIKAN: backward residual
            if self.use_residual and i > 0:
                proj = self.residuals[i]
                if proj is not None:
                    dres = proj.backward(dh_before_relu, accumulate=accumulate)
                else:
                    dres = dh_before_relu  # identity
                dh_main = self.layers[i].backward(dh_before_relu, accumulate=accumulate)
                dh = dh_main + dres
            else:
                dh = self.layers[i].backward(dh_before_relu, accumulate=accumulate)

        return loss

    def flush(self):
        for l in self.layers: l.flush()
        for r in self.residuals:
            if r is not None: r.flush()
        self.out_layer.flush()

    def get_log_probs(self, X):
        self.set_training(False)
        logits = self.forward(X) / max(self.temperature, 1e-6)
        p = _softmax(logits)
        return np.log(p + 1e-30)

    def save(self, path):
        data = {
            "w"  : [(l.W.copy(), l.b.copy()) for l in self.layers],
            "ln" : [(l.ln.gamma.copy(), l.ln.beta.copy()) if l.use_ln
                    else (None, None) for l in self.layers],
            "out": (self.out_layer.W.copy(), self.out_layer.b.copy()),
            "res": [(r.W.copy() if r is not None else None)
                    for r in self.residuals],
            "temperature" : self.temperature,
            "use_residual": self.use_residual,
        }
        np.save(path, data, allow_pickle=True)

    def load(self, path):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W; self.layers[i].b = b
        if "ln" in d:
            for i, (g, bt) in enumerate(d["ln"]):
                if g is not None and self.layers[i].use_ln:
                    self.layers[i].ln.gamma = g
                    self.layers[i].ln.beta  = bt
        if "out" in d:
            W, b = d["out"]
            self.out_layer.W = W; self.out_layer.b = b
        if "res" in d:
            for i, rw in enumerate(d["res"]):
                if rw is not None and self.residuals[i] is not None:
                    self.residuals[i].W = rw
        if "temperature" in d:
            self.temperature = d["temperature"]
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────

def get_lr(epoch, no_improve, cfg, lr_init):
    if epoch <= cfg["warmup_epoch"]:
        alpha = (epoch - 1) / max(cfg["warmup_epoch"] - 1, 1)
        return cfg["lr_warmup"] + alpha * (lr_init - cfg["lr_warmup"])
    return max(
        lr_init * (cfg["lr_decay"] ** (no_improve // cfg["lr_patience"])),
        cfg["lr_min"]
    )


# ─────────────────────────────────────────────────────────────
#  TRAINING LOOP — dengan augmentasi on-the-fly
# ─────────────────────────────────────────────────────────────

def run_training(model, train_ids, val_ids, sequences, targets,
                 cfg, lr_init, max_epochs, early_stop,
                 model_save_path=None, verbose=True,
                 use_augment=True):
    rng_train = np.random.default_rng(cfg["random_seed"])
    rng_aug   = np.random.default_rng(cfg["random_seed"] + 1)
    hist      = []; best = float("inf"); pat = 0; no_improve = 0
    grad_clip = cfg["grad_clip"]; batch_size = cfg["batch_size"]

    if verbose:
        print(f"\n  {'Ep':>4}  {'LR':>8}  {'TrLoss':>8}  "
              f"{'VaLoss':>8}  {'Blank%':>7}  {'t':>6}")
        print(f"  {'-'*56}")

    for epoch in range(1, max_epochs+1):
        t0  = time.time()
        lr  = get_lr(epoch, no_improve, cfg, lr_init)
        model.set_lr(lr); model.set_training(True)

        perm  = rng_train.permutation(len(train_ids))
        order = [train_ids[i] for i in perm]
        total_loss = 0.0; n_batches = 0

        for start in range(0, len(order), batch_size):
            batch = order[start:start+batch_size]
            bl = 0.0; valid = 0
            for aid in batch:
                feat = sequences[aid]
                # PERBAIKAN: augmentasi on-the-fly saat training
                if use_augment:
                    feat = augment_feature(feat, cfg, rng_aug)
                try:
                    lv = model.ctc_step(feat, targets[aid],
                                        grad_clip=grad_clip, accumulate=True)
                    if lv > 0: bl += lv; valid += 1
                except Exception:
                    continue
            model.flush()
            if valid > 0: total_loss += bl / valid; n_batches += 1

        tr_loss = total_loss / max(n_batches, 1)

        model.set_training(False)
        vl = 0.0; vn = 0
        for aid in val_ids:
            try:
                logits = model.forward(sequences[aid])
                lv, _  = ctc_loss_and_grad(logits, targets[aid])
                if lv > 0: vl += lv; vn += 1
            except Exception:
                continue
        val_loss = vl / max(vn, 1)

        blank_pct = _blank_pct(model, val_ids[:15], sequences)
        elapsed   = time.time() - t0
        hist.append({
            "epoch"     : epoch,
            "lr"        : round(lr, 8),
            "train_loss": round(tr_loss, 6),
            "val_loss"  : round(val_loss, 6),
            "blank_pct" : round(blank_pct, 1),
        })

        if verbose:
            print(f"  {epoch:4d}  {lr:8.6f}  {tr_loss:8.4f}  "
                  f"{val_loss:8.4f}  {blank_pct:6.1f}%  {elapsed:5.0f}s")
            if blank_pct > 97.0 and epoch > cfg["warmup_epoch"]:
                print(f"  ⚠ Blank collapse ({blank_pct:.1f}%) — "
                      f"pertimbangkan naikkan temperature")

        if val_loss < best - 1e-5:
            best = val_loss; pat = 0; no_improve = 0
            if model_save_path: model.save(model_save_path)
        else:
            pat += 1; no_improve += 1
            if pat >= early_stop:
                if verbose:
                    print(f"\n  ✓ Early stop ep {epoch} (best={best:.4f})")
                break

    return best, hist


def _blank_pct(model, ids, sequences, n=10):
    model.set_training(False)
    tb = tf = 0
    for aid in ids[:n]:
        lp    = model.get_log_probs(sequences[aid])
        preds = np.argmax(lp, axis=1)
        tb   += (preds == BLANK).sum()
        tf   += len(preds)
    return tb / max(tf, 1) * 100


# ─────────────────────────────────────────────────────────────
#  CTC BEAM SEARCH DECODE
# ─────────────────────────────────────────────────────────────

def ctc_beam_decode_tokens(log_probs: np.ndarray,
                           id2phn: dict,
                           beam_width: int = 10) -> list:
    T, V = log_probs.shape
    NEG  = -float('inf')
    beam = {(): (0.0, NEG)}

    for t in range(T):
        new_beam = {}
        for seq, (pb, pnb) in beam.items():
            lp_b = log_probs[t, BLANK]
            p0, p1 = new_beam.get(seq, (NEG, NEG))
            new_beam[seq] = (np.logaddexp(p0,
                                          np.logaddexp(pb, pnb) + lp_b), p1)
            for c in range(1, V):
                lp_c = log_probs[t, c]
                nseq = seq + (c,)
                if seq and seq[-1] == c:
                    ep_nb = pb + lp_c
                else:
                    ep_nb = np.logaddexp(pb, pnb) + lp_c
                p0n, p1n = new_beam.get(nseq, (NEG, NEG))
                new_beam[nseq] = (p0n, np.logaddexp(p1n, ep_nb))

        beam = dict(
            sorted(new_beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]),
                   reverse=True)[:beam_width]
        )

    best_seq = max(beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]))[0]
    tokens = [
        id2phn[int(c)]
        for c in best_seq
        if id2phn.get(int(c), '<blank>') != '<blank>'
    ]
    return tokens


# ─────────────────────────────────────────────────────────────
#  AKURASI FONEM + CONFUSION MATRIX — DIPERBAIKI
#
#  PERBAIKAN: Confusion matrix per token fonem atomik
#  untuk identifikasi error sistematis
# ─────────────────────────────────────────────────────────────

def compute_phoneme_accuracy_detailed(model, eval_ids, sequences, phn_refs,
                                       id2phn, beam_width=10):
    """
    Return:
      accuracy_pct, per_pct,
      confusion: dict { (ref_tok, hyp_tok): count }
    """
    model.set_training(False)
    total_ref  = 0
    total_edit = 0
    confusion  = defaultdict(int)

    for aid in eval_ids:
        lp       = model.get_log_probs(sequences[aid])
        hyp_toks = ctc_beam_decode_tokens(lp, id2phn, beam_width)
        ref_toks = phn_refs.get(aid, [])

        ref_clean = [t for t in ref_toks if t not in (SILENCE_TOKEN, '<sp>')]
        hyp_clean = [t for t in hyp_toks if t not in (SILENCE_TOKEN, '<sp>')]

        R = len(ref_clean)
        if R == 0: continue
        H = len(hyp_clean)

        # Levenshtein dengan traceback untuk confusion matrix
        dp = np.zeros((R+1, H+1), dtype=np.int32)
        dp[:, 0] = np.arange(R+1)
        dp[0, :] = np.arange(H+1)
        for i in range(1, R+1):
            for j in range(1, H+1):
                cost = 0 if ref_clean[i-1] == hyp_clean[j-1] else 1
                dp[i, j] = min(dp[i-1, j]+1,
                               dp[i, j-1]+1,
                               dp[i-1, j-1]+cost)

        # Traceback untuk confusion
        i, j = R, H
        while i > 0 or j > 0:
            if i > 0 and j > 0:
                cost = 0 if ref_clean[i-1] == hyp_clean[j-1] else 1
                if dp[i, j] == dp[i-1, j-1] + cost:
                    r_tok = ref_clean[i-1]
                    h_tok = hyp_clean[j-1] if j > 0 else '<del>'
                    confusion[(r_tok, h_tok)] += 1
                    i -= 1; j -= 1
                elif dp[i, j] == dp[i-1, j] + 1:
                    confusion[(ref_clean[i-1], '<del>')] += 1
                    i -= 1
                else:
                    confusion[('<ins>', hyp_clean[j-1])] += 1
                    j -= 1
            elif i > 0:
                confusion[(ref_clean[i-1], '<del>')] += 1; i -= 1
            else:
                confusion[('<ins>', hyp_clean[j-1])] += 1; j -= 1

        total_ref  += R
        total_edit += int(dp[R, H])

    if total_ref == 0:
        return 0.0, 100.0, confusion
    per = total_edit / total_ref * 100.0
    acc = max(0.0, 100.0 - per)
    return round(acc, 4), round(per, 4), dict(confusion)


def confusion_to_df(confusion: dict) -> pd.DataFrame:
    rows = []
    for (ref, hyp), cnt in sorted(confusion.items(),
                                   key=lambda x: -x[1]):
        rows.append({"ref": ref, "hyp": hyp,
                     "count": cnt,
                     "type": "correct" if ref == hyp else
                             "substitution" if ref not in ('<ins>',) and
                                              hyp not in ('<del>',) else
                             "deletion" if hyp == '<del>' else "insertion"})
    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────
#  WER — DIPERBAIKI
#
#  PERBAIKAN: WER di-cap 100% (konvensi standar literatur ASR)
# ─────────────────────────────────────────────────────────────

def levenshtein_words(ref: list, hyp: list) -> tuple:
    R, H = len(ref), len(hyp)
    dp   = np.zeros((R+1, H+1), dtype=np.int32)
    dp[:, 0] = np.arange(R+1)
    dp[0, :] = np.arange(H+1)
    for i in range(1, R+1):
        for j in range(1, H+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i, j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+cost)
    i, j = R, H; S = D = I = 0
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            if dp[i,j] == dp[i-1,j-1] + cost:
                if cost: S += 1
                i -= 1; j -= 1
            elif dp[i,j] == dp[i-1,j] + 1:
                D += 1; i -= 1
            else:
                I += 1; j -= 1
        elif i > 0:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    N = max(R, 1)
    # PERBAIKAN: WER di-cap 100%
    wer = min((S + D + I) / N, 1.0)
    return wer, S, D, I, N


def evaluate_wer(model, eval_ids, sequences, word_refs, phn_refs,
                 id2phn, beam_width=10):
    model.set_training(False)
    wer_rows = []; tS = tD = tI = tN = 0

    for aid in eval_ids:
        lp        = model.get_log_probs(sequences[aid])
        hyp_toks  = ctc_beam_decode_tokens(lp, id2phn, beam_width)
        hyp_words = tokens_to_words(hyp_toks)
        hyp_text  = phoneme_tokens_to_text(hyp_toks)
        ref_words = word_refs.get(aid, [])

        wer_val, S, D, I, N = levenshtein_words(ref_words, hyp_words)
        tS += S; tD += D; tI += I; tN += N

        wer_rows.append({
            "audio_id" : aid,
            "WER"      : round(wer_val * 100, 4),
            "S": S, "D": D, "I": I, "N": N,
            "referensi": " ".join(ref_words),
            "hipotesis": hyp_text,
        })

    # PERBAIKAN: WER corpus juga di-cap
    wer_corpus = min((tS+tD+tI) / max(tN,1), 1.0) * 100
    wl = [r["WER"] for r in wer_rows]
    summary = {
        "wer_corpus": round(wer_corpus, 4),
        "wer_mean"  : round(float(np.mean(wl)), 4),
        "wer_min"   : round(float(np.min(wl)), 4),
        "wer_max"   : round(float(np.max(wl)), 4),
        "total_S"   : tS, "total_D": tD, "total_I": tI, "total_N": tN,
        "n_audio"   : len(eval_ids),
    }
    return wer_rows, summary


def print_eval_stats(summ_wer, phn_acc_pct, phn_per_pct, split_name="test"):
    print(f"\n── Evaluasi [{split_name}] {'─'*46}")
    print(f"  Jumlah audio     : {summ_wer['n_audio']}")
    print(f"  WER Corpus       : {summ_wer['wer_corpus']:.2f}% (capped 100%)")
    print(f"  WER Mean         : {summ_wer['wer_mean']:.2f}%")
    print(f"  WER Min / Max    : {summ_wer['wer_min']:.2f}% / {summ_wer['wer_max']:.2f}%")
    print(f"  Akurasi Fonem    : {phn_acc_pct:.2f}%")
    print(f"  PER (Phone Err)  : {phn_per_pct:.2f}%")
    print(f"  Total S/D/I/N    : {summ_wer['total_S']}/"
          f"{summ_wer['total_D']}/{summ_wer['total_I']}/{summ_wer['total_N']}")
    print(f"{'─'*60}")


# ─────────────────────────────────────────────────────────────
#  HYPERPARAMETER TUNING — BAYESIAN OPTIMIZATION (Optuna)
#
#  PERBAIKAN UTAMA:
#  1. Bayesian Optimization (ganti grid search 243 kombinasi)
#  2. Composite score: wer_weight*WER + loss_weight*val_loss
#  3. Sample val di-shuffle tiap trial (anti-bias seleksi)
#  4. Pruning eksperimen buruk lebih awal
# ─────────────────────────────────────────────────────────────

def _composite_score(wer_val, val_loss, cfg):
    """
    Composite score untuk seleksi model terbaik (semakin kecil semakin baik).
    - wer_val  : float, persentase WER (0–100)
    - val_loss : float, CTC validation loss (positif)

    WER dinormalisasi ke [0,1] dengan dibagi 100.
    val_loss dibagi dengan konstanta referensi (5.0) agar skalanya setara.
    Score ↓ = model lebih baik.
    """
    wer_norm  = wer_val / 100.0          # [0, 1]
    loss_norm = val_loss / 5.0           # normalisasi skala referensi
    return cfg["wer_weight"] * wer_norm + cfg["loss_weight"] * loss_norm


def hyperparameter_tuning_bayesian(sequences, targets, word_refs, phn_refs,
                                    vocab, id2phn, actual_dim,
                                    train_ids, val_ids, cfg):
    """
    Bayesian Optimization via Optuna.
    Fallback ke random search jika Optuna tidak tersedia.
    """
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        OPTUNA_OK = True
    except ImportError:
        OPTUNA_OK = False
        print("  ⚠ Optuna tidak tersedia — menggunakan random search.")
        print("    Install: pip install optuna")

    n_trials  = cfg["optuna_trials"]
    results   = []
    best_score = float("inf")
    best_combo = None
    rng_main   = np.random.default_rng(cfg["random_seed"])

    def run_trial(params, trial_idx, rng_trial):
        n_layers = params["n_layers"]
        h_dim    = params["hidden_dim"]
        lr       = params["learning_rate"]
        drop     = params["dropout"]
        ctx_type = params["context_type"]
        temp     = params["temperature"]
        contexts = CONTEXT_PRESETS[n_layers][ctx_type]

        # PERBAIKAN: sample val berbeda tiap trial (anti-bias)
        sample_size = min(60, len(val_ids))
        sample_ids  = list(rng_trial.choice(
            val_ids, sample_size, replace=False))

        print(f"\n[{trial_idx:>3}/{n_trials}] "
              f"n_layers={n_layers}  hidden={h_dim}  "
              f"lr={lr:.0e}  drop={drop}  ctx={ctx_type}  temp={temp:.1f}")

        np.random.seed(cfg["random_seed"] + trial_idx)
        model = TDNNModel(
            input_dim    = actual_dim,
            hidden_dim   = h_dim,
            contexts     = contexts,
            output_dim   = len(vocab),
            dropout      = drop,
            lr           = cfg["lr_warmup"],
            temperature  = temp,
            use_residual = True,
        )

        best_vl, _ = run_training(
            model, train_ids, val_ids, sequences, targets,
            cfg, lr_init=lr,
            max_epochs=cfg["epochs_tuning"],
            early_stop=cfg["early_stop"],
            model_save_path=None,
            verbose=True,
            use_augment=True,
        )

        model.set_training(False)
        _, wer_s = evaluate_wer(
            model, sample_ids, sequences, word_refs, phn_refs,
            id2phn, cfg["beam_width"])
        phn_acc, phn_per, _ = compute_phoneme_accuracy_detailed(
            model, sample_ids, sequences, phn_refs, id2phn, cfg["beam_width"])

        wer_val = wer_s["wer_corpus"]
        score   = _composite_score(wer_val, best_vl, cfg)

        row = {
            "trial"           : trial_idx,
            "n_layers"        : n_layers,
            "hidden_dim"      : h_dim,
            "learning_rate"   : lr,
            "dropout"         : drop,
            "context_type"    : ctx_type,
            "temperature"     : temp,
            "context_detail"  : str(contexts),
            "best_val_loss"   : round(best_vl, 6),
            "wer_val_pct"     : round(wer_val, 2),
            "phoneme_acc_val" : round(phn_acc, 2),
            "per_val"         : round(phn_per, 2),
            "composite_score" : round(score, 6),
        }
        print(f"  → val_loss={best_vl:.4f}  WER={wer_val:.1f}%  "
              f"PhAcc={phn_acc:.1f}%  Score={score:.4f}")
        return row, score, contexts

    print(f"\n{'='*70}")
    print(f"  HYPERPARAMETER TUNING — "
          f"{'BAYESIAN (Optuna)' if OPTUNA_OK else 'RANDOM SEARCH'}")
    print(f"  Trials: {n_trials} | Epoch/trial: {cfg['epochs_tuning']}")
    print(f"  Metrik seleksi: {cfg['wer_weight']}×WER + "
          f"{cfg['loss_weight']}×val_loss (↓)")
    print(f"{'='*70}")

    if OPTUNA_OK:
        import optuna

        def objective(trial):
            params = {
                "n_layers"     : trial.suggest_categorical("n_layers", [3, 4, 5]),
                "hidden_dim"   : trial.suggest_categorical("hidden_dim", [128, 256, 512]),
                "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
                "dropout"      : trial.suggest_float("dropout", 0.1, 0.4),
                "context_type" : trial.suggest_categorical("context_type",
                                    ["narrow", "standard", "wide"]),
                "temperature"  : trial.suggest_float("temperature", 0.8, 2.0),
            }
            rng_trial = np.random.default_rng(cfg["random_seed"] + trial.number)
            row, score, contexts = run_trial(params, trial.number + 1, rng_trial)
            results.append(row)
            return score

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=cfg["random_seed"]),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5,
                                               n_warmup_steps=3),
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_t  = study.best_trial
        n_layers  = best_t.params["n_layers"]
        ctx_type  = best_t.params["context_type"]
        best_combo = {
            "n_layers"    : n_layers,
            "hidden_dim"  : best_t.params["hidden_dim"],
            "learning_rate": best_t.params["learning_rate"],
            "dropout"     : best_t.params["dropout"],
            "context_type": ctx_type,
            "temperature" : best_t.params["temperature"],
            "contexts"    : CONTEXT_PRESETS[n_layers][ctx_type],
        }

    else:
        # Random Search fallback
        for trial_idx in range(1, n_trials + 1):
            rng_trial = np.random.default_rng(cfg["random_seed"] + trial_idx)
            params = {
                "n_layers"     : int(rng_trial.choice([3, 4, 5])),
                "hidden_dim"   : int(rng_trial.choice([128, 256, 512])),
                "learning_rate": float(10 ** rng_trial.uniform(-5, -3)),
                "dropout"      : float(rng_trial.uniform(0.1, 0.4)),
                "context_type" : str(rng_trial.choice(["narrow", "standard", "wide"])),
                "temperature"  : float(rng_trial.uniform(0.8, 2.0)),
            }
            row, score, contexts = run_trial(params, trial_idx, rng_trial)
            results.append(row)
            if score < best_score:
                best_score = score
                best_combo = {**params, "contexts": contexts}

    df_tune = (pd.DataFrame(results)
               .sort_values("composite_score")
               .reset_index(drop=True))
    df_tune.to_csv(OUT_TUNING_CSV, index=False)
    print(f"\n  HasilTuning_TDNN.csv tersimpan ({len(results)} trials)")
    _print_tuning_table(df_tune)
    print(f"\n  Config terbaik → {best_combo}")
    return best_combo, df_tune


def _print_tuning_table(df):
    best_score = df["composite_score"].min()
    print(f"\n{'='*96}")
    print(f"  RINGKASAN HASIL TUNING (15 terbaik — composite score)")
    print(f"{'='*96}")
    print(f"  {'#':>4}  {'nL':>3}  {'hDim':>5}  {'LR':>7}  {'Drop':>5}  "
          f"{'Ctx':>8}  {'Temp':>5}  {'ValLoss':>8}  {'WER%':>7}  "
          f"{'PhAcc%':>7}  {'Score':>8}")
    print(f"  {'-'*90}")
    for _, r in df.head(15).iterrows():
        mark = " ←" if r["composite_score"] == best_score else ""
        print(f"  {int(r['trial']):4d}  "
              f"{int(r['n_layers']):3d}  "
              f"{int(r['hidden_dim']):5d}  "
              f"{r['learning_rate']:7.1e}  "
              f"{r['dropout']:5.2f}  "
              f"{r['context_type']:>8}  "
              f"{r['temperature']:5.2f}  "
              f"{r['best_val_loss']:8.4f}  "
              f"{r['wer_val_pct']:6.1f}%  "
              f"{r['phoneme_acc_val']:6.1f}%  "
              f"{r['composite_score']:8.4f}{mark}")
    print(f"{'='*96}")


# ─────────────────────────────────────────────────────────────
#  K-FOLD CROSS-VALIDATION
#
#  PERBAIKAN: Estimasi performa lebih reliable pada dataset kecil
# ─────────────────────────────────────────────────────────────

def run_kfold_cv(sequences, targets, word_refs, phn_refs,
                 vocab, id2phn, actual_dim, all_ids, best_params, cfg):
    k      = cfg["n_cv_folds"]
    folds  = kfold_splits(all_ids, k, cfg["random_seed"])
    cv_rows = []

    print(f"\n{'='*70}")
    print(f"  K-FOLD CROSS VALIDATION ({k} folds)")
    print(f"{'='*70}")

    for fold_idx, (train_ids, val_ids) in enumerate(folds, 1):
        print(f"\n  ── Fold {fold_idx}/{k} ─────────────────────────────────")
        print(f"     train: {len(train_ids)}  val: {len(val_ids)}")

        # Normalisasi per fold (hanya dari train fold)
        mean_f, std_f = compute_global_norm(sequences, train_ids)
        seq_fold = apply_global_norm(sequences, mean_f, std_f)

        np.random.seed(cfg["random_seed"] + fold_idx)
        model = TDNNModel(
            input_dim    = actual_dim,
            hidden_dim   = best_params["hidden_dim"],
            contexts     = best_params["contexts"],
            output_dim   = len(vocab),
            dropout      = best_params["dropout"],
            lr           = cfg["lr_warmup"],
            temperature  = best_params.get("temperature", 1.0),
            use_residual = True,
        )

        best_vl, _ = run_training(
            model, train_ids, val_ids, seq_fold, targets,
            cfg, lr_init=best_params["learning_rate"],
            max_epochs=cfg["epochs_full"] // 2,
            early_stop=cfg["early_stop_full"],
            verbose=True,
            use_augment=True,
        )

        _, wer_s = evaluate_wer(
            model, val_ids, seq_fold, word_refs, phn_refs,
            id2phn, cfg["beam_width"])
        phn_acc, phn_per, _ = compute_phoneme_accuracy_detailed(
            model, val_ids, seq_fold, phn_refs, id2phn, cfg["beam_width"])

        cv_rows.append({
            "fold"          : fold_idx,
            "n_train"       : len(train_ids),
            "n_val"         : len(val_ids),
            "val_loss"      : round(best_vl, 6),
            "wer_corpus_pct": round(wer_s["wer_corpus"], 2),
            "wer_mean_pct"  : round(wer_s["wer_mean"], 2),
            "phoneme_acc_pct": round(phn_acc, 2),
            "per_pct"       : round(phn_per, 2),
        })
        print(f"  Fold {fold_idx}: WER={wer_s['wer_corpus']:.2f}%  "
              f"PhAcc={phn_acc:.2f}%")

    df_cv = pd.DataFrame(cv_rows)
    df_cv.to_csv(OUT_CV_CSV, index=False)

    # Summary
    print(f"\n  {'─'*50}")
    print(f"  CV Summary ({k} folds):")
    print(f"  WER   : {df_cv['wer_corpus_pct'].mean():.2f}% "
          f"± {df_cv['wer_corpus_pct'].std():.2f}%")
    print(f"  PhAcc : {df_cv['phoneme_acc_pct'].mean():.2f}% "
          f"± {df_cv['phoneme_acc_pct'].std():.2f}%")
    print(f"  PER   : {df_cv['per_pct'].mean():.2f}% "
          f"± {df_cv['per_pct'].std():.2f}%")
    return df_cv


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(FIXED_CONFIG["random_seed"])

    print("="*70)
    print("  TDNN + CTC  Javanese ASR  [IMPROVED]")
    print("="*70)

    # ── TAHAP 1: Load Data ───────────────────────────────────
    print("\n" + "="*70)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC")
    print("="*70)

    (sequences_raw, targets, word_refs, phn_refs,
     phn2id, id2phn, vocab, actual_dim) = load_data(CSV_PATH)

    all_ids = list(sequences_raw.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids, FIXED_CONFIG)
    print(f"\n  Split → train:{len(train_ids)}  val:{len(val_ids)}  "
          f"test:{len(test_ids)}")

    # PERBAIKAN: normalisasi hanya dari train_ids (validasi eksplisit)
    mean_g, std_g = compute_global_norm(sequences_raw, train_ids)
    np.save(NORM_PATH, {"mean": mean_g, "std": std_g})
    sequences = apply_global_norm(sequences_raw, mean_g, std_g)
    print(f"  Normalisasi selesai. Val/test tidak ikut dalam statistik.")

    # ── TAHAP 2: Hyperparameter Tuning ──────────────────────
    print("\n" + "="*70)
    print("  TAHAP 2 — Hyperparameter Tuning (Bayesian/Random Search)")
    print("="*70)

    best_params, df_tune = hyperparameter_tuning_bayesian(
        sequences, targets, word_refs, phn_refs,
        vocab, id2phn, actual_dim,
        train_ids, val_ids, FIXED_CONFIG
    )

    # ── TAHAP 3: Training Penuh ──────────────────────────────
    print("\n" + "="*70)
    print("  TAHAP 3 — Training Penuh dengan Config Terbaik")
    print("="*70)
    for k in ["n_layers", "hidden_dim", "learning_rate",
              "dropout", "context_type", "temperature"]:
        if k in best_params:
            print(f"  {k:<16}: {best_params[k]}")
    print(f"  max_epochs    : {FIXED_CONFIG['epochs_full']}")

    np.random.seed(FIXED_CONFIG["random_seed"])
    best_model = TDNNModel(
        input_dim    = actual_dim,
        hidden_dim   = best_params["hidden_dim"],
        contexts     = best_params["contexts"],
        output_dim   = len(vocab),
        dropout      = best_params["dropout"],
        lr           = FIXED_CONFIG["lr_warmup"],
        temperature  = best_params.get("temperature", 1.0),
        use_residual = True,
    )

    _, history = run_training(
        best_model, train_ids, val_ids, sequences, targets,
        FIXED_CONFIG,
        lr_init    = best_params["learning_rate"],
        max_epochs = FIXED_CONFIG["epochs_full"],
        early_stop = FIXED_CONFIG["early_stop_full"],
        model_save_path = MODEL_BEST_PATH,
        verbose    = True,
        use_augment = True,
    )
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  HasilTraining_TDNN.csv tersimpan")

    if os.path.exists(MODEL_BEST_PATH):
        best_model.load(MODEL_BEST_PATH)

    # ── TAHAP 4: Evaluasi ────────────────────────────────────
    print("\n" + "="*70)
    print("  TAHAP 4 — Evaluasi Model Terbaik")
    print("="*70)

    all_wer_rows   = []
    all_confusion  = defaultdict(int)

    for split_name, split_ids in [("val", val_ids), ("test", test_ids)]:
        wr_rows, summ_wer = evaluate_wer(
            best_model, split_ids, sequences, word_refs, phn_refs,
            id2phn, beam_width=FIXED_CONFIG["beam_width"])
        phn_acc, phn_per, confusion = compute_phoneme_accuracy_detailed(
            best_model, split_ids, sequences, phn_refs,
            id2phn, beam_width=FIXED_CONFIG["beam_width"])

        print_eval_stats(summ_wer, phn_acc, phn_per, split_name)
        all_wer_rows.extend(wr_rows)

        for k_pair, cnt in confusion.items():
            all_confusion[k_pair] += cnt

    # Simpan confusion matrix
    df_conf = confusion_to_df(dict(all_confusion))
    df_conf.to_csv(OUT_CONFUSION_CSV, index=False)

    # Top substitution errors
    subs = df_conf[df_conf["type"] == "substitution"].head(15)
    if not subs.empty:
        print(f"\n  Top substitusi fonem (ref → hyp):")
        for _, row in subs.iterrows():
            print(f"    {row['ref']:>6} → {row['hyp']:<6}  ({row['count']:4d}x)")

    df_w = pd.DataFrame(all_wer_rows)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV, index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(
        OUT_DEC_CSV, index=False)

    # ── TAHAP 5: K-Fold Cross Validation ────────────────────
    print("\n" + "="*70)
    print("  TAHAP 5 — K-Fold Cross Validation")
    print("="*70)
    df_cv = run_kfold_cv(
        sequences_raw, targets, word_refs, phn_refs,
        vocab, id2phn, actual_dim, all_ids, best_params, FIXED_CONFIG
    )

    # ── Ringkasan File Output ────────────────────────────────
    print(f"\n  File tersimpan:")
    for p in [OUT_TUNING_CSV, OUT_WER_CSV, OUT_DEC_CSV,
              OUT_HISTORY_CSV, OUT_CV_CSV, OUT_CONFUSION_CSV,
              MODEL_BEST_PATH]:
        print(f"    {p}")

    return best_model, history, df_tune, df_cv


if __name__ == "__main__":
    model, history, tuning_results, cv_results = main()

  TDNN + CTC  Javanese ASR  [IMPROVED]

  TAHAP 1 — Memuat Data & Ekstraksi MFCC
  Membaca D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv ...
  Shape CSV : (1749, 4)
  Vocab fonem : 31 fonem + 1 blank = 32 unit
  Token khusus: ['<sp>', 'DH', 'NG', 'NY', 'TH']
  Mengekstrak MFCC dari 1749 file .wav ...
    [200/1749] diproses, 0 dilewati
    [400/1749] diproses, 0 dilewati
    [600/1749] diproses, 0 dilewati
    [800/1749] diproses, 1 dilewati
    [1000/1749] diproses, 1 dilewati
    [1200/1749] diproses, 1 dilewati
    [1400/1749] diproses, 1 dilewati
    [1600/1749] diproses, 1 dilewati
    [1749/1749] diproses, 1 dilewati

  Audio berhasil : 1748 | dilewati: 1
    - too_short: 1
  Panjang fonem  : min=7, mean=44.7, max=94

  Split → train:1398  val:174  test:176
  Menghitung normalisasi dari 1398 train samples ...
  Normalisasi OK — mean shape: (39,), mean range: [-246.68, 3.82]
  Normalisasi selesai. Val/test tidak iku

KeyboardInterrupt: 